# PNAD Income Analysis Pipeline

This notebook is the single executable scientific analysis for the project. All numerical procedures are implemented in the documented `pnad_income` package; the notebook orchestrates the pipeline and displays the complete set of descriptive, distributional, and inequality outputs used to inspect the harmonized PNAD/PNAD Contínua income series.


## 1. Configuration and reproducible execution

The annual analytical records are read from `dados_refined/`. The package maps the stored Portuguese fields `ano` and `renda` to the canonical internal names `year` and `income` without altering the Parquet files. The complete available series is analyzed from 1976 to 2025.


In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd

from pnad_income.pipeline import PipelineConfig, pipeline_overview, run_pipeline
from pnad_income.plotting import (
    plot_ccdf_grid,
    plot_gini_evolution,
    plot_histogram_grid,
    plot_lorenz_grid,
    plot_measure_comparison_grid,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATABASE_PATH = Path(os.environ.get("PNAD_DATABASE_PATH", "../dados_refined")).expanduser()
CONFIG = PipelineConfig(
    database_path=DATABASE_PATH,
    ccdf_base=1.05,
    start_year=1976,
    end_year=2025,
)
CONFIG


## 2. Database loading, validation, and analytical coverage

The pipeline loads all annual records, validates the longitudinal schema, attaches year-specific monetary metadata, and constructs the adjusted income series. The diagnostic table below reports the number of observations, temporal coverage, and CCDF output size before any interpretation of the results.


In [ ]:
results = run_pipeline(CONFIG)
overview = pipeline_overview(results)
display(overview)
display(results.panel.head())


## 3. Annual descriptive statistics

For every available survey year, the table reports the number of observations, positive support, arithmetic mean, median, standard deviation, and Gini coefficient for both nominal and adjusted income where available.


In [ ]:
summary = results.summary
display(summary)


## 4. Annual Gini coefficient

The following figure displays the Gini coefficient for every available survey year in a single temporal series. Because the monetary adjustment is a positive year-specific scalar transformation, the within-year Gini coefficient is invariant to that rescaling.


In [ ]:
fig = plot_gini_evolution(summary, value_col="income")
plt.show()


## 5. Annual income histograms — linear frequency scale

All available survey years are displayed below as small multiples. The 45 years are automatically split into two pages to preserve readability. Each panel is computed directly from the harmonized microdata for that year.


In [ ]:
for fig in plot_histogram_grid(
    results.panel,
    value_col="income",
    bins=60,
    yscale="linear",
    ncols=4,
    max_panels=24,
):
    plt.show()


## 6. Annual income histograms — logarithmic frequency scale

A logarithmic frequency axis reveals the low-frequency upper tail that is compressed in ordinary histograms. The income axis remains in the original annual monetary scale in this diagnostic.


In [ ]:
for fig in plot_histogram_grid(
    results.panel,
    value_col="income",
    bins=60,
    yscale="log",
    ncols=4,
    max_panels=24,
):
    plt.show()


## 7. Complementary cumulative distribution function

For a nonnegative income variable $X$, the empirical complementary cumulative distribution is

$$
\widehat{\overline F}(x)
=
\frac{1}{N}
\sum_{i=1}^{N}
\mathbf{1}(X_i\geq x).
$$

Geometric thresholds are defined over the strictly positive support, whereas finite zero-income observations remain in the denominator $N$. The resulting estimator is therefore unconditional.


In [ ]:
ccdf = results.ccdf_nominal_adjusted
display(ccdf.head(20))


## 8. Annual CCDFs — linear axes

The next two pages display the complete annual CCDF series on ordinary linear axes. This representation preserves the empirical probability scale and is useful for inspecting the bulk of each distribution.


In [ ]:
for fig in plot_ccdf_grid(
    ccdf,
    measure="income",
    transform="linear",
    ncols=4,
    max_panels=24,
):
    plt.show()


## 9. Annual CCDFs — log-log axes

The same annual distributions are shown on logarithmic axes. This representation expands the upper tail and is appropriate for examining approximate scaling regimes and deviations from them.


In [ ]:
for fig in plot_ccdf_grid(
    ccdf,
    measure="income",
    transform="loglog",
    ncols=4,
    max_panels=24,
):
    plt.show()


## 10. Legacy $\ln[\ln(\mathrm{CCDF})]$ diagnostic

The historical analysis also used the transformation $\ln[\ln(\mathrm{CCDF}[\%])]$. It is retained here as a legacy diagnostic to reproduce the earlier exploratory figures. Because the transformation depends on expressing the CCDF in percentage units, it should not be confused with the scale-invariant log-log representation above.


In [ ]:
for fig in plot_ccdf_grid(
    ccdf,
    measure="income",
    transform="double_log",
    ncols=4,
    max_panels=24,
):
    plt.show()


## 11. Annual Lorenz curves

The Lorenz curve for every survey year is plotted below. Each panel compares cumulative income share with cumulative population share; the diagonal corresponds to perfect equality.


In [ ]:
for fig in plot_lorenz_grid(
    results.panel,
    value_col="income",
    ncols=4,
    max_panels=24,
):
    plt.show()


## 12. Nominal versus adjusted distributions for all years

The monetary standardization rescales each annual distribution to the common reference used in the project. The following panels compare nominal and adjusted CCDFs year by year on log-log axes.


In [ ]:
for fig in plot_measure_comparison_grid(
    ccdf,
    measures=("income", "income_adj"),
    transform="loglog",
    ncols=4,
    max_panels=24,
):
    plt.show()


## 13. Effective-income availability

The current refined release contains the harmonized longitudinal income series only. If a future release includes a separate effective-income field, the pipeline can construct the corresponding annual distribution. The diagnostic below records whether such observations are presently available.


In [ ]:
ccdf_effective = results.ccdf_habitual_effective
if ccdf_effective.empty:
    print("The current refined database does not contain usable income_effective observations.")
else:
    display(ccdf_effective.head(20))


## 14. Final data-quality diagnostics

The final table reports missingness and numerical support for the central analytical measures. It should be reviewed after any change to the refined database or harmonization metadata.


In [ ]:
diagnostic_columns = [
    c for c in ("income", "income_adj", "income_effective", "income_effective_adj")
    if c in results.panel.columns
]
diagnostics = pd.DataFrame({
    "column": diagnostic_columns,
    "non_missing": [int(results.panel[c].notna().sum()) for c in diagnostic_columns],
    "missing": [int(results.panel[c].isna().sum()) for c in diagnostic_columns],
    "minimum": [results.panel[c].min() for c in diagnostic_columns],
    "maximum": [results.panel[c].max() for c in diagnostic_columns],
})
display(diagnostics)


## 15. Reusable result objects

The validated objects available after execution are `results.panel`, `results.summary`, `results.ccdf_nominal_adjusted`, and `results.ccdf_habitual_effective`. Subsequent analyses should consume these objects rather than recreate preprocessing or distributional calculations inside notebook cells.
